# 04. PINN Final Evaluation

**v6 Section 17.10.1** | Phase C Development Notebook #4

## Purpose
학습 완료된 PINN 모델의 정밀 검증

## When to use
`scripts/train_phase_c.py` 학습 완료 후, `checkpoints/phase_c_final.pt` 로드

## Checks (v6 Section 18)
1. BM boundary: |U| < 0.05 at BM regions
2. ASM matching: PINN vs ASM < 10% error at z=40 slit
3. Interior fringe: std > 0.1 (not uniform)
4. Design sensitivity: |U| varies with delta, w
5. PSF 7-pixel calculation
6. Multi-angle response (theta=0, 15, 30, 40)

In [ ]:
import sys
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml
%matplotlib inline

from backend.core.pinn_model import PurePINN
from backend.training.loss_functions import ASMIncidentLUT
from backend.training.red_flag_detector import detect_red_flags
from backend.physics.boundary_conditions import compute_is_bm

DEVICE = torch.device('cpu')
print('Ready')

---
## Cell 1: Load Final Model

In [ ]:
# ── Load checkpoint ──
CKPT_PATH = PROJECT_ROOT / 'checkpoints' / 'phase_c_final.pt'

if not CKPT_PATH.exists():
    # Fallback: find latest checkpoint
    ckpts = sorted((PROJECT_ROOT / 'checkpoints').glob('phase_c_*.pt'),
                   key=lambda p: p.stat().st_mtime, reverse=True)
    if ckpts:
        CKPT_PATH = ckpts[0]
        print(f'Final not found, using: {CKPT_PATH.name}')
    else:
        raise FileNotFoundError('No checkpoints found. Run train_phase_c.py first.')

# Find config
exp_dirs = sorted((PROJECT_ROOT / 'experiments').glob('*phase_c*'),
                  key=lambda p: p.stat().st_mtime, reverse=True)
if exp_dirs and (exp_dirs[0] / 'config.yaml').exists():
    with open(exp_dirs[0] / 'config.yaml') as f:
        cfg = yaml.safe_load(f)
    mcfg = cfg['model']
else:
    mcfg = {'hidden_dim': 128, 'num_layers': 4, 'num_freqs': 48, 'omega_0': 30.0}

model = PurePINN(**mcfg).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params, epoch {ckpt["epoch"]}')
print(f'Checkpoint: {CKPT_PATH.name}')

asm_lut = ASMIncidentLUT(str(PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'))

---
## Cell 2: BM Boundary Verification

v6 Section 18: BM 영역에서 |U| < 0.05

In [ ]:
N = 1000
x_e = torch.linspace(0, 504, N)

def make_coords(z_val, delta1=0, delta2=0, w1=10, w2=10, theta=0):
    sin_t = np.sin(np.radians(theta))
    cos_t = np.cos(np.radians(theta))
    return torch.stack([
        x_e, torch.full((N,), float(z_val)),
        torch.full((N,), float(delta1)), torch.full((N,), float(delta2)),
        torch.full((N,), float(w1)), torch.full((N,), float(w2)),
        torch.full((N,), sin_t), torch.full((N,), cos_t),
    ], dim=1)

def get_amplitude(z_val, **kw):
    with torch.no_grad():
        U = model(make_coords(z_val, **kw))
        return torch.sqrt(U[:,0]**2 + U[:,1]**2).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (z_val, bm_name, d_idx, w_idx) in zip(axes, [(20, 'BM1', 0, 10), (40, 'BM2', 0, 10)]):
    amp = get_amplitude(z_val)
    is_bm = compute_is_bm(x_e, torch.zeros(N), torch.full((N,), 10.0))
    
    ax.plot(x_e.numpy(), amp, 'b-', lw=0.8)
    ax.fill_between(x_e.numpy(), 0, amp, where=is_bm.numpy(), alpha=0.3, color='red', label='BM region')
    ax.fill_between(x_e.numpy(), 0, amp, where=~is_bm.numpy(), alpha=0.3, color='green', label='Slit')
    ax.axhline(y=0.05, color='red', ls='--', lw=1, label='Target < 0.05')
    
    bm_amp = amp[is_bm.numpy()]
    slit_amp = amp[~is_bm.numpy()]
    passed = bm_amp.mean() < 0.05
    
    ax.set_title(f'{bm_name} (z={z_val}) | BM mean={bm_amp.mean():.4f} | '
                 f'{"PASS" if passed else "FAIL"}', fontsize=11,
                 color='#2E7D32' if passed else '#C62828')
    ax.set_xlabel('x (um)')
    ax.set_ylabel('|U|')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Check 1: BM Boundary (|U| should be ~0 in red regions)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 3: ASM Matching at z=40

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, theta in zip(axes.flat, [0, 15, 30, 40]):
    amp_pinn = get_amplitude(40, theta=theta)
    sin_t = np.sin(np.radians(theta))
    U_re, U_im = asm_lut.lookup(x_e, torch.full((N,), sin_t))
    amp_asm = torch.sqrt(U_re**2 + U_im**2).numpy()
    
    # Only compare inside slits
    is_slit = ~compute_is_bm(x_e, torch.zeros(N), torch.full((N,), 10.0))
    slit_mask = is_slit.numpy()
    
    if slit_mask.sum() > 0:
        rel_err = np.abs(amp_pinn[slit_mask] - amp_asm[slit_mask]) / (amp_asm[slit_mask] + 1e-8)
        mean_err = rel_err.mean() * 100
    else:
        mean_err = 0
    
    ax.plot(x_e.numpy(), amp_pinn, 'b-', lw=0.8, label='PINN')
    ax.plot(x_e.numpy(), amp_asm, 'r--', lw=0.8, label='ASM target')
    ax.set_title(f'theta={theta} | slit rel.error={mean_err:.1f}% | '
                 f'{"PASS" if mean_err < 10 else "FAIL"} (<10%)',
                 fontsize=10, color='#2E7D32' if mean_err < 10 else '#C62828')
    ax.set_xlabel('x (um)')
    ax.set_ylabel('|U|')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Check 2: ASM Matching at z=40 (inside slits)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 4: Interior Fringe & 2D Field

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cross-sections at multiple z
for ax, z_val in zip(axes.flat[:3], [30, 20, 10]):
    amp = get_amplitude(z_val)
    ax.plot(x_e.numpy(), amp, 'b-', lw=0.8)
    ax.set_title(f'z={z_val} | std={amp.std():.4f} | '
                 f'{"GOOD" if amp.std() > 0.1 else "Needs more training"}',
                 fontsize=10)
    ax.set_xlabel('x (um)')
    ax.set_ylabel('|U|')
    ax.grid(True, alpha=0.2)
    for i in range(7):
        ax.axvline(x=i*72+36, color='gray', ls=':', alpha=0.2)

# 2D field map
ax = axes[1, 1]
z_r = torch.linspace(0, 40, 100)
x_r = torch.linspace(0, 504, 200)
Z, X = torch.meshgrid(z_r, x_r, indexing='ij')
Nm = Z.numel()
with torch.no_grad():
    c = torch.stack([X.flatten(), Z.flatten(),
                     torch.zeros(Nm), torch.zeros(Nm),
                     torch.full((Nm,), 10.0), torch.full((Nm,), 10.0),
                     torch.zeros(Nm), torch.ones(Nm)], dim=1)
    U = model(c)
    amp_map = torch.sqrt(U[:,0]**2 + U[:,1]**2).reshape(100, 200).numpy()

im = ax.imshow(amp_map, aspect='auto', origin='lower', extent=[0,504,0,40], cmap='hot')
ax.axhline(y=20, color='cyan', ls='--', alpha=0.7)
ax.axhline(y=40, color='cyan', ls='-', alpha=0.7)
ax.set_title('|U|(x,z) 2D field', fontsize=10)
ax.set_xlabel('x (um)')
ax.set_ylabel('z (um)')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Check 3: Interior Fringe (std > 0.1 = diffraction learned)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 5: Design Variable Sensitivity

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sweeps = [
    ('delta_bm1', np.linspace(-10, 10, 7), 'delta1'),
    ('delta_bm2', np.linspace(-10, 10, 7), 'delta2'),
    ('w1', np.linspace(5, 20, 7), 'w1'),
    ('w2', np.linspace(5, 20, 7), 'w2'),
]

for ax, (param_name, values, kw_name) in zip(axes.flat, sweeps):
    for val in values:
        kwargs = {'delta1': 0, 'delta2': 0, 'w1': 10, 'w2': 10}
        kwargs[kw_name] = val
        amp = get_amplitude(0, **kwargs)  # z=0 (OPD)
        ax.plot(x_e.numpy(), amp, lw=0.6, label=f'{val:.0f}', alpha=0.8)
    
    ax.set_title(f'{param_name} sweep at z=0 (OPD)', fontsize=10)
    ax.set_xlabel('x (um)')
    ax.set_ylabel('|U|')
    ax.legend(fontsize=7, ncol=4, title=param_name)
    ax.grid(True, alpha=0.2)

plt.suptitle('Check 4: Design Variable Sensitivity (curves should differ!)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 6: PSF 7-Pixel Calculation

In [ ]:
import math

def compute_psf_7(delta1=0, delta2=0, w1=10, w2=10, theta=0, n_samples=200):
    """Compute PSF for 7 OPD pixels (v6 Section 3.5.4)."""
    PITCH = 72.0
    OPD_W = 10.0
    psf = np.zeros(7)
    sin_t = math.sin(math.radians(theta))
    cos_t = math.cos(math.radians(theta))
    
    with torch.no_grad():
        for i in range(7):
            cx = i * PITCH + PITCH / 2
            x = torch.linspace(cx - OPD_W/2, cx + OPD_W/2, n_samples)
            coords = torch.stack([
                x, torch.zeros(n_samples),
                torch.full((n_samples,), float(delta1)),
                torch.full((n_samples,), float(delta2)),
                torch.full((n_samples,), float(w1)),
                torch.full((n_samples,), float(w2)),
                torch.full((n_samples,), sin_t),
                torch.full((n_samples,), cos_t),
            ], dim=1)
            U = model(coords)
            intensity = (U[:,0]**2 + U[:,1]**2).numpy()
            psf[i] = intensity.mean() * OPD_W
    return psf

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, theta in zip(axes.flat, [0, 15, 30, 40]):
    psf = compute_psf_7(theta=theta)
    bars = ax.bar(range(7), psf, color=['#EF5350' if i%2==0 else '#42A5F5' for i in range(7)],
                  edgecolor='white', lw=1.5)
    ax.set_xlabel('OPD pixel index')
    ax.set_ylabel('PSF (intensity)')
    ax.set_title(f'PSF @ theta={theta} | center={psf[3]:.4f}', fontsize=10)
    ax.set_xticks(range(7))
    ax.set_xticklabels(['R','V','R','V(c)','R','V','R'], fontsize=8)
    ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('Check 5: PSF 7-Pixel (center pixel should be dominant at theta=0)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 7: Final Red Flag Check & Summary

In [ ]:
report = detect_red_flags(model, DEVICE)

print('='*60)
print('PHASE C EVALUATION SUMMARY')
print('='*60)
print(f'Model: {n_params:,} params, epoch {ckpt["epoch"]}')
print(f'Checkpoint: {CKPT_PATH.name}')
print()
print('Red Flag Report:')
print(report.summary())
print()
print(f'Interior CoV:        {report.interior_cov:.4f} (want > 0.05)')
print(f'BM1 mean |U|:        {report.bm1_mean_amp:.4f} (want < 0.05)')
print(f'BM2 mean |U|:        {report.bm2_mean_amp:.4f} (want < 0.05)')
print(f'Design sensitivity:  {report.design_sensitivity:.4f} (want > 0.01)')
print()

all_pass = (not report.has_red_flag and not report.has_warning)
if all_pass:
    print('RESULT: ALL CHECKS PASSED - Ready for Phase D (FNO distillation)')
elif report.has_red_flag:
    print('RESULT: RED FLAG - Do NOT proceed. Diagnose and retrain.')
else:
    print('RESULT: WARNINGS - May need more training or tuning.')
print('='*60)